# Fit Multiple Files with Shared Parameters

This notebook fits multiple datasets simultaneously, sharing some parameters
across all files while fitting others independently per file.

**Example System:**
- 6 datasets with identical spectral model (GLP + linear background)
- Different peak shift amplitudes across files (varying pump fluence)
- Shared time dependence: exponential shift of peak position (with IRF)

**Key Concept:**
Parameters are classified by `vary` level in the YAML model:
- `"project"`: shared across all files (fitted once)
- `"file"`: independent per file (fitted separately for each)
- `"static"`: fixed (set from baseline, not fitted)

**Workflow:**
1. Load all datasets into a single Project
2. Fit baseline for each file
3. Load 2D models with project/file/static vary levels and<br>
   run `project.fit_2d()` to fit all files simultaneously
4. Compare fit results to known truth used to create synthetic data

In [ ]:
from pathlib import Path

import numpy as np
import trspecfit

## 1. Load Data

Create a Project and load all 6 datasets. Each file shares the same energy and
time axes but has a different peak shift amplitude (`expFun A`).

In [ ]:
project = trspecfit.Project(path=Path.cwd())

data_folder = 'data'
energy = np.loadtxt(project.path / data_folder / 'energy.csv')
time = np.loadtxt(project.path / data_folder / 'time.csv')

shift_amplitudes = [5, 2, 1, 0.5, 0.2, 0.1]
files = []

for i, amp in enumerate(shift_amplitudes, start=1):
    file_data = np.loadtxt(project.path / data_folder / f'data_{i}.csv', delimiter=',')
    f = trspecfit.File(
        parent_project=project,
        path=f'data_{i}',
        data=file_data,
        energy=energy,
        time=time,
    )
    files.append(f)
    print(f"File {i}: expFun A = {amp}, shape {file_data.shape}")

project.describe(detail=1)

## 2. Fit Baseline Spectra

### 2.1. Define Fitting Region and Baseline

Set energy and time limits, then define the baseline (ground state) spectrum
for all files at once.

In [ ]:
# Set fitting limits (same for all files)
project.set_fit_limits(
    energy_limits=[5, 18],
    time_limits=[-20, 99],
)

# Define baseline from early time points (before dynamics onset)
project.define_baselines(time_start=0, time_stop=22, time_type='ind')

### 2.2. Fit Baseline

Load a baseline model and fit it on every file. Since the spectral shape is
identical across files (only the time-dependent shift differs), the baseline
parameters should converge to similar values.

In [ ]:
# Load baseline model onto all files
project.load_models(model_yaml='models_energy.yaml', model_info='base')

# Fit baselines (try_ci=0 skips confidence intervals: see 12_uncertainty_mcmc)
project.fit_baselines(model_name='base', stages=2, try_ci=0)

## 3. Global 2D Fitting

Load the 2D model, add time dependence, and fit all files simultaneously.

The `vary` levels control parameter sharing:

**Energy model** (`models_energy.yaml`, all `"static"`):
- All energy parameters fixed from baseline fit

**Time model** (`models_time.yaml`):
- `"project"`: `tau`, `gaussCONV SD` — shared dynamics across all files
- `"file"`: `expFun A` — independent shift amplitude per file

In [ ]:
# Load 2D model onto all files
project.load_models(model_yaml='models_energy.yaml', model_info='2D')

# Add time dependence to peak position
project.add_time_dependences(
    target_model='2D',
    target_parameter='GLP_01_x0',
    dynamics_yaml='models_time.yaml',
    dynamics_model='MonoExpPosIRF',
)

# Fit all files simultaneously
project.fit_2d(model_name='2D', stages=2)

## 4. Shared vs. Per-File Parameters

The `vary` levels are the whole point — confirm them in the fit output.
`file.get_fit_results(fit_type="2d")` returns one row per parameter; below we
collect the three dynamics parameters for every file into one table.

- **`SD` and `tau` are `"project"`** — fit once, shared across all files, so
  their columns are *identical* down every row.
- **`A` is `"file"`** — fit independently per file, so its column varies row to
  row and tracks the known shift amplitudes.

Truth: shared `SD = 3`, `tau = 50`; per-file `A = [5, 2, 1, 0.5, 0.2, 0.1]`
(`data/models_time_truth.yaml`).

In [ ]:
import pandas as pd

# vary level per parameter: "project" = shared (fit once), "file" = independent
rows = []
for f, A_truth in zip(files, shift_amplitudes):
    vals = f.get_fit_results(fit_type="2d").set_index("name")["value"]
    rows.append(
        {
            "file": f.name,
            "SD (project)": vals["GLP_01_x0_gaussCONV_SD"],
            "tau (project)": vals["GLP_01_x0_expFun_01_tau"],
            "A (file)": vals["GLP_01_x0_expFun_01_A"],
            "A truth": A_truth,
        }
    )

pd.DataFrame(rows).set_index("file").round(3)

## Tips

- **`vary` levels live in the YAML, not the code.** `project` / `file` / `static` on each parameter decide what is shared, fit per-file, or frozen from the baseline — reshape the sharing scheme by editing `models_time.yaml`, no code change.
- **`project.fit_2d()` is a true joint fit.** Every file enters one residual; `project` parameters resolve to a single shared value, `file` parameters to one per file. Contrast `20_multi_file_independent_fit`, which fits each file separately.
- **Static energy parameters come from the baseline.** The energy model is all `static`, so the ground-state shape is frozen per file and only the time dynamics are fit in 2D.

## Next Steps

- [`20_multi_file_independent_fit`](../20_multi_file_independent_fit/example.ipynb) — the independent-fit counterpart, no shared parameters.
- [`12_uncertainty_mcmc`](../12_uncertainty_mcmc/example.ipynb) — put error bars on these shared and per-file parameters.